# 👔 MobileNetV3-Small Staff Classifier — Purplle Retail

This notebook trains a **MobileNetV3-Small** binary classifier (staff vs customer) on crops extracted from your CCTV clips, then exports to **ONNX**.

## Prerequisites
1. **Runtime → Change runtime type → GPU (T4)**
2. Complete Notebook 1 first (or have frames already extracted)
3. Your CCTV clips must include staff members wearing identifiable uniforms

## What gets produced
- `mobilenet_staff.onnx` → drop into `models/`
- Target: precision ≥ 0.90, recall ≥ 0.85

## Design Reference
From `design.md §2.5`: Two-signal ensemble — MobileNetV3 + HSV histogram.
The pipeline uses ONNX output for the NN signal and computes HSV Bhattacharyya distance at runtime.

In [ ]:
# ============================================================
# CELL 1 — Verify GPU and install dependencies
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU — switch runtime')

!pip install -q torch torchvision opencv-python-headless scikit-learn matplotlib
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# ──────────────────────────────────────────────────────────────
# ⚙️  CONFIGURE: Path to your Stores/ directory in Drive
# ──────────────────────────────────────────────────────────────
CLIPS_DIR = '/content/drive/MyDrive/purplle/Stores'   # <-- change this

clips = []
for root, dirs, files in os.walk(CLIPS_DIR):
    for f in files:
        if f.endswith(('.mp4', '.avi', '.mov')):
            clips.append(os.path.join(root, f))

print(f'Found {len(clips)} clips')
for c in clips:
    print(' ', os.path.basename(c))

In [ ]:
# ============================================================
# CELL 3 — Configure staff crop extraction
#
# MANUAL STEP: You need to identify which frames / timestamps
# contain staff members. The cells below help you:
#   A) Preview frames to identify staff time ranges
#   B) Auto-extract crops using YOLOv11m
#   C) Manually sort crops into staff/ and customer/ folders
# ============================================================

# ──────────────────────────────────────────────────────────────
# ⚙️  CONFIGURE: Which clips are most likely to show staff?
# Staff are typically visible in billing_area and zone clips.
# The pipeline uses staff wearing Purplle uniforms (identifiable colour).
# ──────────────────────────────────────────────────────────────

# Approximate time ranges (seconds) where staff are visible per clip
# Set to None to extract all frames
STAFF_TIME_HINTS = {
    'billing_area.mp4': None,    # Scan entire billing clip for staff
    'CAM 5 - billing.mp4': None, # Store 1 billing
    'zone.mp4': None,
    'CAM 1 - zone.mp4': None,
}

print('Configuration set.')
print('The next cell will extract person crops from these clips.')

In [ ]:
# ============================================================
# CELL 4 — Extract person crops using YOLOv11m
# ============================================================
from ultralytics import YOLO
import cv2, os, numpy as np

!pip install -q ultralytics

model_det = YOLO('yolo11m.pt')

CROPS_DIR = '/content/crops/all'
os.makedirs(CROPS_DIR, exist_ok=True)

SAMPLE_EVERY_N_FRAMES = 15    # At 15fps → 1 crop per second per track
MIN_CROP_HEIGHT = 80          # Reject tiny crops (far-away people)
crop_count = 0

for clip_path in clips:
    clip_name = os.path.splitext(os.path.basename(clip_path))[0].replace(' ', '_')
    cap = cv2.VideoCapture(clip_path)
    idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % SAMPLE_EVERY_N_FRAMES != 0:
            idx += 1
            continue

        results = model_det(frame, conf=0.5, classes=[0], verbose=False)
        for ri, result in enumerate(results):
            for bi, box in enumerate(result.boxes):
                x1, y1, x2, y2 = [int(c) for c in box.xyxy[0].tolist()]
                crop = frame[y1:y2, x1:x2]
                if crop.shape[0] < MIN_CROP_HEIGHT:
                    continue
                fname = f'{clip_name}_f{idx:06d}_p{bi}.jpg'
                cv2.imwrite(os.path.join(CROPS_DIR, fname), crop)
                crop_count += 1
        idx += 1

    cap.release()
    print(f'  {clip_name}: done')

print(f'\nTotal crops extracted: {crop_count}')
print(f'Saved to: {CROPS_DIR}')

In [ ]:
# ============================================================
# CELL 5 — Preview crops to identify staff
# Run this to see a grid of crops — look for Purplle uniform colours
# ============================================================
import glob
import matplotlib.pyplot as plt
import cv2, random

all_crops = glob.glob('/content/crops/all/*.jpg')
random.shuffle(all_crops)
preview = all_crops[:24]

fig, axes = plt.subplots(4, 6, figsize=(18, 12))
for ax, crop_path in zip(axes.flat, preview):
    img = cv2.cvtColor(cv2.imread(crop_path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (128, 256))
    ax.imshow(img)
    ax.set_title(os.path.basename(crop_path)[:20], fontsize=6)
    ax.axis('off')
plt.suptitle('Person Crops — Identify staff uniforms', fontsize=14)
plt.tight_layout()
plt.savefig('/content/crop_preview.png', dpi=100)
plt.show()
print('Preview saved to /content/crop_preview.png')

In [ ]:
# ============================================================
# CELL 6 — Sort crops into staff / customer folders
#
# OPTION A (Manual): Run this cell, then use the file browser
#   to move staff crops into /content/crops/staff/
#   and customer crops into /content/crops/customer/
#
# OPTION B (HSV-based auto-sort): Uses dominant colour heuristic
#   to pre-sort crops. Review and fix misclassifications.
#
# ──────────────────────────────────────────────────────────────
# ⚙️  CONFIGURE: Staff uniform dominant HSV range (Purplle purple)
# Purplle staff uniforms are typically purple/magenta.
# Adjust H range after viewing crops in Cell 5.
# HSV Hue: 0-180 in OpenCV. Purple ≈ 120-160, Magenta ≈ 160-180/0-10
# ──────────────────────────────────────────────────────────────
import cv2, os, glob, shutil, numpy as np

STAFF_HUE_LOW  = 120   # Lower bound of staff uniform hue
STAFF_HUE_HIGH = 160   # Upper bound of staff uniform hue
STAFF_SAT_MIN  = 60    # Minimum saturation (avoid grey/white)
STAFF_FRAC_THR = 0.20  # >20% pixels in hue range → likely staff

for split in ['staff', 'customer']:
    os.makedirs(f'/content/crops/{split}', exist_ok=True)

all_crops = glob.glob('/content/crops/all/*.jpg')
auto_staff = 0
auto_customer = 0

for crop_path in all_crops:
    img = cv2.imread(crop_path)
    if img is None:
        continue
    # Focus on upper body (top 60% = torso region where uniform is visible)
    h, w = img.shape[:2]
    torso = img[:int(h*0.6), :]
    hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv,
        np.array([STAFF_HUE_LOW, STAFF_SAT_MIN, 50]),
        np.array([STAFF_HUE_HIGH, 255, 255]))
    frac = mask.sum() / (255.0 * mask.size)

    if frac >= STAFF_FRAC_THR:
        shutil.copy(crop_path, f'/content/crops/staff/{os.path.basename(crop_path)}')
        auto_staff += 1
    else:
        shutil.copy(crop_path, f'/content/crops/customer/{os.path.basename(crop_path)}')
        auto_customer += 1

print(f'Auto-sorted → Staff: {auto_staff}, Customer: {auto_customer}')
print()
print('⚠️  IMPORTANT: Review the staff/ folder and move any misclassified')
print('   customer crops out (and vice versa) before training.')
print('   Target: ≥50 staff crops, ≥200 customer crops for a balanced dataset.')

In [ ]:
# ============================================================
# CELL 7 — Balance dataset and create train/val splits
# ============================================================
import glob, os, random, shutil

random.seed(42)

staff_crops    = glob.glob('/content/crops/staff/*.jpg')
customer_crops = glob.glob('/content/crops/customer/*.jpg')

print(f'Staff crops:    {len(staff_crops)}')
print(f'Customer crops: {len(customer_crops)}')

# Balance: downsample majority class to 3x minority
min_count = min(len(staff_crops), len(customer_crops))
max_count = min(min_count * 3, max(len(staff_crops), len(customer_crops)))

random.shuffle(staff_crops)
random.shuffle(customer_crops)
staff_crops    = staff_crops[:max_count]
customer_crops = customer_crops[:max_count]

print(f'\nAfter balancing:')
print(f'  Staff: {len(staff_crops)}, Customer: {len(customer_crops)}')

for split in ['train', 'val']:
    for cls in ['staff', 'customer']:
        os.makedirs(f'/content/dataset_cls/{split}/{cls}', exist_ok=True)

def split_and_copy(crop_list, class_name, train_ratio=0.8):
    split_idx = int(len(crop_list) * train_ratio)
    for i, path in enumerate(crop_list):
        split = 'train' if i < split_idx else 'val'
        shutil.copy(path, f'/content/dataset_cls/{split}/{class_name}/{os.path.basename(path)}')

split_and_copy(staff_crops, 'staff')
split_and_copy(customer_crops, 'customer')

for split in ['train', 'val']:
    for cls in ['staff', 'customer']:
        n = len(glob.glob(f'/content/dataset_cls/{split}/{cls}/*.jpg'))
        print(f'  {split}/{cls}: {n} images')

In [ ]:
# ============================================================
# CELL 8 — Build DataLoaders with augmentation
# ============================================================
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Augmentation strategy from design.md §8.2
train_transform = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder('/content/dataset_cls/train', transform=train_transform)
val_dataset   = datasets.ImageFolder('/content/dataset_cls/val',   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

print(f'Classes: {train_dataset.classes}')  # ['customer', 'staff']
print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')

# Class index mapping (important for interpreting model output)
CLASS_TO_IDX = train_dataset.class_to_idx
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
STAFF_CLASS_IDX = CLASS_TO_IDX['staff']
print(f'Staff class index: {STAFF_CLASS_IDX}')

In [ ]:
# ============================================================
# CELL 9 — Build MobileNetV3-Small model
# From design.md §8.2: Replace classifier head with Linear(576, 2)
# ============================================================
import torch
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load pretrained MobileNetV3-Small
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)

# Replace classifier head: 576 → 2 (customer / staff)
# MobileNetV3-Small last adaptive pooling output: 576 features
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, 2)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable:,}')

In [ ]:
# ============================================================
# CELL 10 — Training loop
# From design.md §8.2: 30 epochs, AdamW lr=1e-3, cosine annealing
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=30)

EPOCHS = 30
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_acc = correct / total
    scheduler.step()

    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss / len(val_loader))
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/mobilenet_staff_best.pth')

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
              f'Train Loss: {history["train_loss"][-1]:.4f} | '
              f'Val Loss: {history["val_loss"][-1]:.4f} | '
              f'Val Acc: {val_acc:.4f}')

print(f'\nBest val accuracy: {best_val_acc:.4f}')

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'],   label='Val Loss')
axes[0].legend(); axes[0].set_title('Loss')
axes[1].plot(history['val_acc'], label='Val Accuracy')
axes[1].axhline(y=0.90, color='r', linestyle='--', label='Target (0.90)')
axes[1].legend(); axes[1].set_title('Validation Accuracy')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=100)
plt.show()

In [ ]:
# ============================================================
# CELL 11 — Evaluate precision and recall
# Targets: precision ≥ 0.90, recall ≥ 0.85 (design.md §2.5)
# ============================================================
import torch
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Load best model
model.load_state_dict(torch.load('/content/mobilenet_staff_best.pth'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print('Classification Report:')
print(classification_report(
    all_labels, all_preds,
    target_names=list(IDX_TO_CLASS[i] for i in range(2))
))

# Extract staff class metrics
from sklearn.metrics import precision_recall_fscore_support
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, pos_label=STAFF_CLASS_IDX, average='binary'
)

print(f'Staff Precision: {precision:.4f}  (target: ≥ 0.90)')
print(f'Staff Recall:    {recall:.4f}   (target: ≥ 0.85)')
print(f'Staff F1-Score:  {f1:.4f}')

if precision >= 0.90 and recall >= 0.85:
    print('\n✅  Both targets met! Proceeding to ONNX export.')
else:
    if precision < 0.90:
        print('⚠️  Precision below target. Review staff crops for misclassified customers.')
    if recall < 0.85:
        print('⚠️  Recall below target. Add more staff crop variety (different angles/lighting).')

In [ ]:
# ============================================================
# CELL 12 — Export to ONNX
# Output shape: (1, 2) → [customer_prob, staff_prob]
# The pipeline reads probs[1] as staff probability
# ============================================================
import torch
import torch.nn.functional as F

model.load_state_dict(torch.load('/content/mobilenet_staff_best.pth', map_location='cpu'))
model.eval().cpu()

# Wrap model to output softmax probabilities
class StaffClassifierWithSoftmax(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model

    def forward(self, x):
        return F.softmax(self.model(x), dim=1)

export_model = StaffClassifierWithSoftmax(model)
export_model.eval()

# Export input: (1, 3, 256, 128) — batch of BGR crops resized to 256x128
dummy_input = torch.randn(1, 3, 256, 128)

ONNX_PATH = '/content/mobilenet_staff.onnx'
torch.onnx.export(
    export_model,
    dummy_input,
    ONNX_PATH,
    opset_version=17,
    input_names=['input'],
    output_names=['probs'],   # shape: (batch, 2) — [customer_prob, staff_prob]
    dynamic_axes={'input': {0: 'batch_size'}, 'probs': {0: 'batch_size'}},
)

print(f'ONNX model exported to: {ONNX_PATH}')

# Verify
import onnx
m = onnx.load(ONNX_PATH)
onnx.checker.check_model(m)
print('✅  ONNX model validation passed')
print(f'Output[1] = staff probability (used by pipeline)')

In [ ]:
# ============================================================
# CELL 13 — Benchmark ONNX inference on CPU
# ============================================================
import onnxruntime as ort
import numpy as np
import time

session = ort.InferenceSession('/content/mobilenet_staff.onnx', providers=['CPUExecutionProvider'])

dummy = np.random.rand(1, 3, 256, 128).astype(np.float32)

# Warmup
for _ in range(10):
    session.run(None, {'input': dummy})

# Benchmark
N = 100
t0 = time.perf_counter()
for _ in range(N):
    session.run(None, {'input': dummy})
elapsed = time.perf_counter() - t0
avg_ms = (elapsed / N) * 1000

print(f'Average inference: {avg_ms:.2f} ms per crop')
print(f'Throughput:        {1000/avg_ms:.0f} crops/sec')
print('Target: < 5ms per crop (design.md §2.5)')

if avg_ms < 5:
    print('✅  Meets <5ms target on CPU')
else:
    print('⚠️  Above 5ms target. Model is still usable but may limit pipeline throughput.')

In [ ]:
# ============================================================
# CELL 14 — Save to Google Drive and download
# ============================================================
import shutil, os

drive_dest = '/content/drive/MyDrive/purplle/models/mobilenet_staff.onnx'
os.makedirs(os.path.dirname(drive_dest), exist_ok=True)
shutil.copy('/content/mobilenet_staff.onnx', drive_dest)

print(f'✅  Saved to Google Drive: {drive_dest}')
print()
print('Next steps:')
print('  1. Download mobilenet_staff.onnx from Google Drive')
print('  2. Place it in: models/mobilenet_staff.onnx')
print('  3. Restart the vision pipeline:')
print('     docker compose restart vision-pipeline')
print('  The staff_classifier.py will auto-detect and use the ONNX model.')
print('  Without it, the pipeline falls back to HSV-only classification.')

from google.colab import files
files.download('/content/mobilenet_staff.onnx')

## 📊 Training Summary

| Item | Value |
|------|-------|
| Base model | MobileNetV3-Small (ImageNet pretrained) |
| Classifier head | `Linear(576, 2)` — [customer, staff] |
| Input size | `(1, 3, 256, 128)` — person crop, normalised |
| Output | `softmax([customer_prob, staff_prob])` |
| Target precision | ≥ 0.90 |
| Target recall | ≥ 0.85 |
| Target speed | < 5ms/crop on CPU |
| Output file | `mobilenet_staff.onnx` → place in `models/` |

### How the pipeline uses this model:
```python
# In staff_classifier.py:
outputs = onnx_session.run(None, {input_name: crop_tensor})
staff_prob = outputs[0][0][1]   # Index 1 = staff class
is_staff = (staff_prob > 0.70) AND (bhattacharyya_dist < 0.30)
```

### Troubleshooting low precision:
- Check that staff crops don't include customers in Purplle-coloured clothing
- Increase `STAFF_FRAC_THR` in Cell 6 to reduce false positives in auto-sort
- Add negative examples: customers in similar colours

### Troubleshooting low recall:
- Staff may not always wear full uniform (partial coverage, jackets off)
- Add crops of staff from different angles and lighting conditions
- Lower the `NN_STAFF_THRESHOLD` from 0.70 to 0.60 in `staff_classifier.py`